# 🥈 Silver Layer - Validation & Self-Healing

This notebook validates Bronze datasets, applies automated correction rules,
quarantines invalid records, and prepares cleaned datasets for the Gold layer.

In [0]:
import os

from pyspark.sql.functions import *

from pyspark.sql.types import *

from datetime import datetime

from utils.config import *

In [0]:
BUS_BRONZE_PATH = os.path.join(
    BRONZE_PATH,
    "bus_gps"
)

EMERGENCY_BRONZE_PATH = os.path.join(
    BRONZE_PATH,
    "emergency"
)

print(BUS_BRONZE_PATH)
print(EMERGENCY_BRONZE_PATH)

In [0]:
bus_df = spark.read.parquet(
    BUS_BRONZE_PATH
)

emergency_df = spark.read.parquet(
    EMERGENCY_BRONZE_PATH
)

Data Quality Validation

In [0]:
from pyspark.sql.functions import col, current_timestamp


def validate_missing_zone(df):
    return df.filter(col("zone").isNull())

def validate_invalid_gps(df):
    return df.filter(
        (col("latitude") < -90) |
        (col("latitude") > 90) |
        (col("longitude") < -180) |
        (col("longitude") > 180)
    )

def validate_negative_delay(df):
    return df.filter(
        col("delay_minutes") < 0
    )

def validate_future_timestamp(df):
    return df.filter(
        col("timestamp") > current_timestamp()
    )

def validate_duplicate_records(df):

    return df.groupBy(df.columns)\
             .count()\
             .filter("count > 1")

    return df.join(
        duplicate_ids,
        on="bus_id",
        how="inner"
    )

In [0]:
missing_zone_df = validate_missing_zone(bus_df)
invalid_gps_df = validate_invalid_gps(bus_df)
negative_delay_df = validate_negative_delay(bus_df)
future_timestamp_df = validate_future_timestamp(bus_df)
duplicate_bus_df = validate_duplicate_records(bus_df)

In [0]:
print("=" * 50)
print("Validation Summary")
print("=" * 50)
print("Missing Zone      :", missing_zone_df.count())
print("Invalid GPS       :", invalid_gps_df.count())
print("Negative Delay    :", negative_delay_df.count())
print("Future Timestamp  :", future_timestamp_df.count())
print("Duplicate Records :", duplicate_bus_df.count())
print("=" * 50)

# Emergency Data Quality Validation

In [0]:
def validate_missing_zone_emergency(df):
    return df.filter(col("zone").isNull())

def validate_invalid_severity(df):
    return df.filter(
        (col("severity") < 1) |
        (col("severity") > 5)
    )

def validate_negative_response_time(df):
    return df.filter(
        col("response_time") < 0
    )

def validate_future_timestamp_emergency(df):
    return df.filter(
        col("timestamp") > current_timestamp()
    )

def validate_duplicate_records(df):

    return df.groupBy(df.columns) \
             .count() \
             .filter("count > 1")

In [0]:
missing_zone_emergency_df = validate_missing_zone_emergency(emergency_df)
invalid_severity_df = validate_invalid_severity(emergency_df)
negative_response_df = validate_negative_response_time(emergency_df)
future_timestamp_emergency_df = validate_future_timestamp_emergency(emergency_df)
duplicate_emergency_df = validate_duplicate_records(emergency_df)

In [0]:
print("=" * 50)
print("Emergency Validation Summary")
print("=" * 50)
print("Missing Zone             :", missing_zone_emergency_df.count())
print("Invalid Severity         :", invalid_severity_df.count())
print("Negative Response Time   :", negative_response_df.count())
print("Future Timestamp         :", future_timestamp_emergency_df.count())
print("Duplicate Records        :", duplicate_emergency_df.count())
print("=" * 50)

# Bus GPS Self-Healing

In [0]:
from pyspark.sql.functions import lit

# Create working copy from Bronze
bus_silver_df = bus_df

# Initialize repair tracking columns
bus_silver_df = (
    bus_silver_df
    .withColumn("repair_status", lit("clean"))
    .withColumn("repair_reason", lit(None).cast("string"))
    .withColumn("repair_confidence", lit(100))
)

print("✅ Silver dataset initialized.")

## Repair Missing Zone

In [0]:
from pyspark.sql.functions import (
    col,first,when)

In [0]:
# Find the most common zone for each bus

zone_lookup = (
    bus_silver_df
    .filter(col("zone").isNotNull())
    .groupBy("bus_id")
    .agg(first("zone").alias("correct_zone"))
)

zone_lookup.show(5)

In [0]:
# Join lookup table with bus dataset
bus_silver_df = (
    bus_silver_df
    .join(
        zone_lookup,
        on="bus_id",
        how="left"
    )
)
print("✅ Zone lookup joined.")

In [0]:
# Repair missing zones

bus_silver_df = (
    bus_silver_df
    .withColumn(
        "zone",
        when(
            col("zone").isNull(),
            col("correct_zone")
        ).otherwise(col("zone"))
    )
    .withColumn(
        "repair_status",
        when(
            col("correct_zone").isNotNull(),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("correct_zone").isNotNull(),
            "Missing Zone"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("correct_zone").isNotNull(),
            95
        ).otherwise(col("repair_confidence"))
    )
)

print("✅ Missing Zone repaired.")

In [0]:
bus_silver_df = bus_silver_df.drop("correct_zone")

In [0]:
# Verify repair

print("Remaining Missing Zones :")

bus_silver_df.filter(
    col("zone").isNull()
).count()

## Self-Healing 2 - Invalid GPS Coordinates